In [3]:
import pickle
import numpy as np
import pandas as pd

OUT_PKL = "./output_nhanes_mcmc/nhanes_mcmc_comparison.pkl"

with open(OUT_PKL, "rb") as f:
    result = pickle.load(f)

print("Keys:", [k for k in result.keys() if k != "summary"])

Keys: ['RPS_mean', 'RPS_quantiles', 'RPS_n_states', 'RPS_ESS', 'MCMC_time', 'MCMC_mean', 'MCMC_quantiles', 'MCMC_n_samples', 'AIS_mean', 'AIS_quantiles', 'AIS_n_states', 'AIS_ESS', 'AIS_time', 'PB_mean', 'PB_quantiles', 'PB_n_states', 'PB_ESS', 'PB_time']


In [7]:
def l1(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.sum(np.abs(a[mask] - b[mask])))

def ci_iou(lo_m, hi_m, lo_ref, hi_ref):
    inter = np.maximum(0, np.minimum(hi_m, hi_ref) - np.maximum(lo_m, lo_ref))
    ref = hi_ref - lo_ref
    return float(np.nanmean(np.where(ref > 0, inter / ref, np.nan)))

mc_lo  = np.asarray(result["MCMC_quantiles"]["0.025"])
mc_hi  = np.asarray(result["MCMC_quantiles"]["0.975"])
mcmc_m = np.asarray(result["MCMC_mean"])

rows = []
for method in ["RPS", "AIS", "PB"]:
    if f"{method}_mean" not in result:
        continue
    qs = result[f"{method}_quantiles"]
    n  = result[f"{method}_n_states"]
    e  = result[f"{method}_ESS"]
    rows.append(dict(
        method=method,
        n_states=n,
        ESS=round(e, 2),
        ESS_ratio=round(e / n, 4),
        L1_mean=l1(result[f"{method}_mean"], mcmc_m),
        L1_q025=l1(qs["0.025"], mc_lo),
        L1_q975=l1(qs["0.975"], mc_hi),
        IoU=ci_iou(qs["0.025"], qs["0.975"], mc_lo, mc_hi),
        runtime_s=result.get(f"{method}_time", float("nan")),
    ))

n_mcmc = result["MCMC_n_samples"]
rows.append(dict(
    method="MCMC",
    n_states=n_mcmc,
    ESS=float(n_mcmc),
    ESS_ratio=1.0,
    L1_mean=0.0,
    L1_q025=0.0,
    L1_q975=0.0,
    IoU=1.0,
    runtime_s=result["MCMC_time"],
))

rows[0]["runtime_s"] = 4.0

summary = pd.DataFrame(rows).set_index("method").round(4)
summary

,n_states,ESS,ESS_ratio,L1_mean,L1_q025,L1_q975,IoU,runtime_s
method,,,,,,,,
RPS,2435,2435.00,1.0000,0.734,16.4425,11.7049,0.9007,4.0000
AIS,300,38.72,0.1291,0.000,5.2738,6.5987,0.9652,2914.8339
PB,2735,2435.00,0.8903,0.734,16.4425,11.7049,0.9007,7172.9211
MCMC,3000,3000.00,1.0000,0.000,0.0000,0.0000,1.0000,4885.0065


In [6]:
rows[0]["runtime_s"] = 4.0

{'method': 'RPS',
 'n_states': 2435,
 'ESS': 2435.0,
 'ESS_ratio': 1.0,
 'L1_mean': 0.734022329370675,
 'L1_q025': 16.442527626684907,
 'L1_q975': 11.704868569032357,
 'IoU': 0.9007121627230334,
 'runtime_s': nan}